# Notebook 2 — Cointegration Analysis (Top Pair Deep-Dive)

For the highest-ranked pair this notebook shows:
1. In-sample ADF unit-root test results for each leg
2. Engle-Granger OLS regression and residual ADF test
3. Johansen test output
4. OU half-life estimation and fitted process
5. In-sample vs out-of-sample spread behaviour

All pair-selection decisions were made in Notebook 1. This notebook
is purely diagnostic — it uses the already-selected pair.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pickle

from src.data_loader import load_prices, split_in_out_of_sample
from src.cointegration import (
    adf_unit_root, engle_granger, johansen, estimate_half_life
)

plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})
print('Imports OK')

In [ ]:
# Load data and selected pairs
prices = load_prices()
prices_is, prices_oos = split_in_out_of_sample(prices, in_sample_end='2020-12-31')

with open('../data/selected_pairs.pkl', 'rb') as f:
    selected_pairs = pickle.load(f)

if not selected_pairs:
    raise RuntimeError("No selected pairs found. Run Notebook 1 first.")

sp = selected_pairs[0]  # top-ranked pair
ty, tx = sp.analysis.ticker_y, sp.analysis.ticker_x
print(f"Analysing top pair: {ty} / {tx}")
print(f"  EG p-value:  {sp.eg_p_value:.4f}")
print(f"  Beta (OLS):  {sp.ols_beta:.4f}")
print(f"  Half-life:   {sp.half_life_days:.1f} days")

## 1. Data overview for the top pair

In [ ]:
y_is = prices_is[ty].dropna()
x_is = prices_is[tx].dropna()
common_is = y_is.index.intersection(x_is.index)
y_is, x_is = y_is.loc[common_is], x_is.loc[common_is]

print(f"In-sample observations: {len(common_is)}")
print(f"\n{ty} summary:\n{y_is.describe().round(2)}")
print(f"\n{tx} summary:\n{x_is.describe().round(2)}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(y_is.index, y_is.values, color='steelblue', lw=1.2, label=ty)
axes[0].plot(x_is.index, x_is.values, color='darkorange', lw=1.2, label=tx)
axes[0].set_title(f'{ty} and {tx} — In-Sample Price Levels')
axes[0].set_ylabel('Adj Close (AUD)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Log prices for visual stationarity check
axes[1].plot(y_is.index, np.log(y_is.values), color='steelblue', lw=1.2, label=f'log({ty})')
axes[1].plot(x_is.index, np.log(x_is.values), color='darkorange', lw=1.2, label=f'log({tx})')
axes[1].set_title('Log Prices')
axes[1].set_ylabel('Log Price')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. ADF Unit Root Tests (Gate 1)

In [ ]:
adf_y = adf_unit_root(y_is, ticker=ty)
adf_x = adf_unit_root(x_is, ticker=tx)

print(f"{'Ticker':<12} {'ADF stat':>10} {'p-value':>10} {'I(1)?':>8} {'Diff ADF':>10} {'Diff p':>10}")
print('-' * 65)
for result in [adf_y, adf_x]:
    print(f"{result.ticker:<12} {result.adf_stat:>10.3f} {result.p_value:>10.4f} "
          f"{'Yes' if result.is_integrated_order_1 else 'No':>8} "
          f"{result.diff_adf_stat:>10.3f} {result.diff_p_value:>10.4f}")

print()
print("Interpretation:")
print(f"  {ty}: ADF p={adf_y.p_value:.4f} > 0.05 => fail to reject unit root (non-stationary)")
print(f"  {tx}: ADF p={adf_x.p_value:.4f} > 0.05 => fail to reject unit root (non-stationary)")
print(f"  Both first differences: stationary => both are I(1) ✓")

In [ ]:
# Visualise: price levels vs first differences
fig, axes = plt.subplots(2, 2, figsize=(13, 6))

for col, (ticker, series) in enumerate([(ty, y_is), (tx, x_is)]):
    axes[0, col].plot(series.index, series.values, color='steelblue', lw=1)
    axes[0, col].set_title(f'{ticker} — Level (non-stationary)')
    axes[0, col].grid(True, alpha=0.3)

    diff = series.diff().dropna()
    axes[1, col].plot(diff.index, diff.values, color='darkorange', lw=0.8)
    axes[1, col].set_title(f'{ticker} — First Difference (stationary)')
    axes[1, col].grid(True, alpha=0.3)

plt.suptitle('I(1) Verification: Levels vs First Differences', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Engle-Granger Two-Step Test (Gate 2)

In [ ]:
# Mathematical formulation:
# Step 1: OLS  Y_t = alpha + beta * X_t + e_t
# Step 2: ADF on residuals e_t — if I(0), the pair is cointegrated

eg = engle_granger(y_is, x_is, ticker_y=ty, ticker_x=tx)
print("Engle-Granger (1987) Two-Step Procedure")
print("=" * 50)
print(f"Step 1 — OLS regression: {ty} = alpha + beta * {tx} + e")
print(f"  alpha  = {eg.ols_alpha:.4f}")
print(f"  beta   = {eg.ols_beta:.4f}  (hedge ratio)")
print()
print(f"Step 2 — ADF on residuals:")
print(f"  ADF statistic = {eg.residual_adf_stat:.4f}")
print(f"  ADF p-value   = {eg.residual_p_value:.4f}")
print(f"  EG p-value (MacKinnon 1994 corrected) = {eg.eg_p_value:.4f}")
print()
if eg.is_cointegrated:
    print(f"  RESULT: Reject unit root in residuals (p<0.05) => COINTEGRATED ✓")
else:
    print(f"  RESULT: Fail to reject unit root in residuals (p>0.05) => NOT cointegrated")

In [ ]:
# Plot OLS regression scatter + residuals
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

aligned = pd.concat([y_is, x_is], axis=1).dropna()
y_vals = aligned.iloc[:, 0].values
x_vals = aligned.iloc[:, 1].values

axes[0].scatter(x_vals, y_vals, alpha=0.3, s=8, color='steelblue')
x_line = np.linspace(x_vals.min(), x_vals.max(), 100)
y_line = eg.ols_alpha + eg.ols_beta * x_line
axes[0].plot(x_line, y_line, color='crimson', lw=2, label=f'y = {eg.ols_alpha:.2f} + {eg.ols_beta:.3f}x')
axes[0].set_xlabel(f'{tx} price')
axes[0].set_ylabel(f'{ty} price')
axes[0].set_title(f'OLS: {ty} ~ {tx} (in-sample)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residuals over time
residuals = pd.Series(y_vals - (eg.ols_alpha + eg.ols_beta * x_vals), index=aligned.index)
axes[1].plot(residuals.index, residuals.values, color='steelblue', lw=0.9)
axes[1].axhline(0, color='black', lw=0.8, linestyle='--')
axes[1].axhline(residuals.std() * 2, color='crimson', lw=0.7, linestyle=':', alpha=0.7, label='+2σ')
axes[1].axhline(-residuals.std() * 2, color='crimson', lw=0.7, linestyle=':', alpha=0.7, label='-2σ')
axes[1].set_title(f'OLS Residuals (spread) — ADF p = {eg.residual_p_value:.4f}')
axes[1].set_ylabel('Residual (price units)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Johansen Test (Gate 3)

In [ ]:
joh = johansen(y_is, x_is, ticker_y=ty, ticker_x=tx)
print("Johansen (1988) Maximum Likelihood Cointegration Test")
print("=" * 55)
print(f"Configuration: det_order=0, k_ar_diff=1")
print()
print(f"  Null: rank = 0 (no cointegration)")
print(f"  {'Statistic':<20} {'Test value':>12} {'95% critical':>14} {'Reject H0?':>12}")
print(f"  {'-'*60}")
print(f"  {'Trace':<20} {joh.trace_stat:>12.3f} {joh.trace_crit_95:>14.3f} {'Yes ✓' if joh.trace_stat > joh.trace_crit_95 else 'No':>12}")
print(f"  {'Max-Eigenvalue':<20} {joh.max_eig_stat:>12.3f} {joh.max_eig_crit_95:>14.3f} {'Yes ✓' if joh.max_eig_stat > joh.max_eig_crit_95 else 'No':>12}")
print()
print(f"  Cointegration rank: {joh.cointegration_rank}")
if joh.is_cointegrated:
    print("  RESULT: Johansen confirms cointegration ✓")
else:
    print("  RESULT: Johansen does NOT confirm cointegration")

## 5. OU Half-Life Estimation (Gate 4)

In [ ]:
# Construct in-sample spread and fit OU process
spread_is = y_is - eg.ols_beta * x_is
hl = estimate_half_life(spread_is, ticker_y=ty, ticker_x=tx)

print("Ornstein-Uhlenbeck Half-Life Estimation")
print("=" * 45)
print(f"Model: Δs_t = kappa*(mu - s_{{t-1}}) + sigma*ε_t")
print(f"  (discrete: Δs_t = a + b*s_{{t-1}} + ε_t, b = -kappa*dt)")
print()
print(f"  kappa (mean-reversion speed): {hl.ou_kappa:.4f} per day")
print(f"  mu    (long-run mean):         {hl.ou_mu:.4f}")
print(f"  sigma (diffusion coeff):       {hl.ou_sigma:.4f}")
print()
print(f"  Half-life = ln(2) / kappa = {hl.half_life_days:.2f} trading days")
if hl.is_valid:
    print(f"  RESULT: Half-life in [1, 30] days — VALID ✓")
else:
    print(f"  RESULT: Half-life outside [1, 30] days — would be rejected")

In [ ]:
# Plot spread with OU mean-reversion bands
fig, axes = plt.subplots(2, 1, figsize=(12, 7))

# Upper plot: spread with reversion bands
mu = spread_is.mean()
sigma_band = spread_is.std()
axes[0].plot(spread_is.index, spread_is.values, lw=1, color='steelblue', label='Spread')
axes[0].axhline(mu, color='black', lw=1.2, linestyle='--', label=f'Mean ({mu:.2f})')
axes[0].axhline(mu + 2*sigma_band, color='crimson', lw=0.8, linestyle=':', label='±2σ')
axes[0].axhline(mu - 2*sigma_band, color='crimson', lw=0.8, linestyle=':')
axes[0].set_title(f'In-Sample Spread: {ty} − {eg.ols_beta:.3f}×{tx}  (HL={hl.half_life_days:.1f}d)')
axes[0].set_ylabel('Spread (price units)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Lower plot: ACF of spread to visualise mean-reversion speed
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(spread_is.dropna(), lags=40, alpha=0.05, ax=axes[1], zero=False)
axes[1].set_title('Spread Autocorrelation (should decay quickly for mean-reverting spread)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. In-Sample vs Out-of-Sample Spread Behaviour

In [ ]:
# Apply the IN-SAMPLE estimated hedge ratio to the OUT-OF-SAMPLE period
# This is the honest evaluation: beta was estimated on IS data only
y_oos = prices_oos[ty].dropna() if ty in prices_oos.columns else pd.Series(dtype=float)
x_oos = prices_oos[tx].dropna() if tx in prices_oos.columns else pd.Series(dtype=float)

if len(y_oos) > 0 and len(x_oos) > 0:
    common_oos = y_oos.index.intersection(x_oos.index)
    y_oos, x_oos = y_oos.loc[common_oos], x_oos.loc[common_oos]
    # Apply STATIC in-sample beta (deliberately — to show regime comparison)
    spread_oos = y_oos - eg.ols_beta * x_oos
    
    print(f"In-sample spread:  mean={spread_is.mean():.3f}, std={spread_is.std():.3f}")
    print(f"Out-of-sample spread: mean={spread_oos.mean():.3f}, std={spread_oos.std():.3f}")
    
    # Is the OOS spread still stationary?
    from statsmodels.tsa.stattools import adfuller
    adf_oos_stat, adf_oos_p, *_ = adfuller(spread_oos.dropna(), autolag='AIC')
    print(f"\nOOS spread ADF test: stat={adf_oos_stat:.3f}, p={adf_oos_p:.4f}")
    if adf_oos_p < 0.05:
        print("  Spread remains stationary out-of-sample ✓ (cointegration held)")
    else:
        print("  Spread is NOT stationary out-of-sample — cointegration may have broken")
else:
    print("Insufficient out-of-sample data for comparison.")
    spread_oos = pd.Series(dtype=float)

In [ ]:
# Side-by-side IS vs OOS spread plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (spread, period, color) in zip(axes, [
    (spread_is, 'In-Sample (2015–2020)', 'steelblue'),
    (spread_oos, 'Out-of-Sample (2021–present)', 'darkorange'),
]):
    if len(spread) == 0:
        ax.set_visible(False)
        continue
    z = (spread - spread_is.mean()) / spread_is.std()  # normalise by IS stats (no lookahead)
    ax.plot(z.index, z.values, lw=0.9, color=color)
    ax.axhline(0, color='black', lw=0.8, linestyle='--')
    ax.axhline(2, color='crimson', lw=0.6, linestyle=':', alpha=0.7)
    ax.axhline(-2, color='crimson', lw=0.6, linestyle=':', alpha=0.7)
    ax.set_title(f'{period}\n{ty}/{tx} spread z-score (IS normalisation)')
    ax.set_ylabel('Z-score')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Summary of all pairs' cointegration statistics
print("All selected pairs — cointegration statistics (in-sample)")
print(f"{'Rank':<5} {'Pair':<25} {'EG p':>8} {'HL (d)':>8} {'Beta':>8} {'Johansen':>10}")
print('-' * 70)
for sp in selected_pairs:
    print(f"#{sp.rank:<4} {sp.analysis.ticker_y}/{sp.analysis.ticker_x:<22} "
          f"{sp.eg_p_value:>8.4f} {sp.half_life_days:>8.1f} {sp.ols_beta:>8.3f}")